# Practice 4 — визуомоторный BC на Google Colab (GPU)

Ноутбук для запуска практики **без локальной GPU**: сбор датасета, обучение и оценка выполняются здесь, на бесплатной GPU.

**Перед запуском:** `Runtime → Change runtime type → Hardware accelerator → GPU`.

Это baseline-конфигурация (вход модели — изображение 84×84). По мере прохождения уроков вы расширите код (добавите проприоцепцию) и сможете обучать новые конфигурации тем же ноутбуком.

In [ ]:
!nvidia-smi

## 1. Клонируем проект

Для **baseline** (Урок 2) достаточно оригинального репозитория — ячейка ниже готова к запуску.

Если вы уже внесли свои правки по урокам 3–6 (`env.py`, `collect_data.py`, `model.py`, `train_bc.py`, `inference.py`) — запушьте их в **свой форк** и подставьте его URL вместо оригинального, иначе Colab будет обучать код без ваших изменений.

In [ ]:
# Для правок из уроков 3–6 замените URL на адрес вашего форка
!git clone https://github.com/Yandex-Practicum/ap-physiclaai-4.git
%cd ap-physiclaai-4

## 2. Зависимости

PyTorch на Colab уже установлен. Нужны `timm`, `einops`, `tensorboard` (обучение) и `mujoco` (сбор и rollout).

In [ ]:
!pip -q install timm einops tensorboard mujoco

## 3. Сбор датасета

Опорная политика собирает демонстрации; `collect_data.py` пишет в `.npz` `obs` и `actions`. MuJoCo на Colab — headless через EGL (`MUJOCO_GL=egl`).

In [ ]:
import os; os.environ['MUJOCO_GL'] = 'egl'
!MUJOCO_GL=egl python3 collect_data.py --checkpoint checkpoints/rl_expert.pt \
    --num_episodes 1000 --save_dir dataset/train --only_success --seed 42
!MUJOCO_GL=egl python3 collect_data.py --checkpoint checkpoints/rl_expert.pt \
    --num_episodes 200  --save_dir dataset/eval  --only_success --seed 100

## 4. Обучение baseline

Чекпоинты → `logs/<exp>/checkpoints/`, логи TensorBoard → `logs/<exp>/`.

In [ ]:
!python3 train_bc.py --train_dir dataset/train --eval_dir dataset/eval --exp_name bc_baseline --epochs 100 --batch_size 64 --lr 1e-4

## 5. TensorBoard

Кривые `train/loss` и `eval/loss`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

## 6. Rollout-оценка (Success Rate)

In [ ]:
!MUJOCO_GL=egl python3 inference.py --checkpoint logs/bc_baseline/checkpoints/best.pt --model bc --episodes 50 --seed 999
!MUJOCO_GL=egl python3 inference.py --checkpoint checkpoints/rl_expert.pt --model rl --episodes 50 --seed 999

## 7. Скачать чекпоинт

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('bc_baseline', 'zip', 'logs', 'bc_baseline')
files.download('bc_baseline.zip')